
> ```bash
> mlflow server --backend-store-uri sqlite:///mlflow.db \
>     --default-artifact-root ./mlruns --host 0.0.0.0 --port 5000 --allowed-hosts "*" --cors-allowed-origins "http://localhost:5000, http://127.0.0.1:5000"
> ```


In [3]:
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.datasets import fetch_openml

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mnist-classifier")
print("Tracking URI:", mlflow.get_tracking_uri())

/home/navadeep04/AIops/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/08/30 12:00:53 INFO mlflow.tracking.fluent: Experiment with name 'mnist-classifier' does not exist. Creating a new experiment.


Tracking URI: http://localhost:5000


In [4]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False)

X = mnist.data
y = mnist.target

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def train_and_evaluate(
    hidden_layer_sizes=(128,),
    learning_rate=0.001,
    max_iter=20):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        learning_rate_init=learning_rate,
        max_iter=max_iter,
        random_state=42,)

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="macro")

    return model, acc, f1


# Sanity check — no MLflow involved yet
_, acc, f1 = train_and_evaluate()

print(f"accuracy={acc:.4f}  f1_macro={f1:.4f}")

accuracy=0.9544  f1_macro=0.9539


/home/navadeep04/AIops/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


In [8]:
def train_and_log(hidden_layer_sizes=(128,),learning_rate=0.001,max_iter=20,run_name=None):
    with mlflow.start_run(run_name=run_name):
        # --- parameters ---
        mlflow.log_param("hidden_layer_sizes", hidden_layer_sizes)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_iter", max_iter)
        # Train and evaluate MLP
        model, acc, f1 = train_and_evaluate(
            hidden_layer_sizes,
            learning_rate,
            max_iter)
        # --- metrics ---
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_macro", f1)

        mlflow.set_tag("team", "data-science")
        # Log model
        mlflow.sklearn.log_model(model,name="model",skops_trusted_types=
                                 ["sklearn.neural_network._stochastic_optimizers.AdamOptimizer"])

        run_id = mlflow.active_run().info.run_id

        print(
            f"Logged run {run_id}  |  "
            f"acc={acc:.4f}  f1={f1:.4f}"
        )

        return run_id


baseline_run_id = train_and_log(
    hidden_layer_sizes=(128,),
    learning_rate=0.001,
    max_iter=20,
    run_name="mlp-baseline"
)

/home/navadeep04/AIops/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run 805ecf0bbf1d40a7aa4ef096e5570833  |  acc=0.9544  f1=0.9539
🏃 View run mlp-baseline at: http://localhost:5000/#/experiments/1/runs/805ecf0bbf1d40a7aa4ef096e5570833
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [9]:
sweep_run_ids = []

for hidden_size in [32, 64, 256]:
    for lr in [0.001,0.01,0.1]:

        rid = train_and_log(
            hidden_layer_sizes=(hidden_size,),
            learning_rate=lr,
            max_iter=20,
            run_name=f"mlp-hidden-{hidden_size}")

        sweep_run_ids.append(rid)

print("Sweep run IDs:", sweep_run_ids)

/home/navadeep04/AIops/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run 81460da545c24a63bcea2352c08fdd86  |  acc=0.9319  f1=0.9309
🏃 View run mlp-hidden-32 at: http://localhost:5000/#/experiments/1/runs/81460da545c24a63bcea2352c08fdd86
🧪 View experiment at: http://localhost:5000/#/experiments/1
Logged run 7d8b6da57ba74ba48dea8523dca6553f  |  acc=0.6401  f1=0.6209
🏃 View run mlp-hidden-32 at: http://localhost:5000/#/experiments/1/runs/7d8b6da57ba74ba48dea8523dca6553f
🧪 View experiment at: http://localhost:5000/#/experiments/1
Logged run cfe4c4277e01430b9bd35284005139cb  |  acc=0.0997  f1=0.0181
🏃 View run mlp-hidden-32 at: http://localhost:5000/#/experiments/1/runs/cfe4c4277e01430b9bd35284005139cb
🧪 View experiment at: http://localhost:5000/#/experiments/1


/home/navadeep04/AIops/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run eda80189780f42b3a5684683ab130b2e  |  acc=0.9456  f1=0.9451
🏃 View run mlp-hidden-64 at: http://localhost:5000/#/experiments/1/runs/eda80189780f42b3a5684683ab130b2e
🧪 View experiment at: http://localhost:5000/#/experiments/1
Logged run 46e3047b7faf466d815766ee60ee916d  |  acc=0.8373  f1=0.8515
🏃 View run mlp-hidden-64 at: http://localhost:5000/#/experiments/1/runs/46e3047b7faf466d815766ee60ee916d
🧪 View experiment at: http://localhost:5000/#/experiments/1
Logged run 080ec2daf36446f88340bd006a0dec6d  |  acc=0.1143  f1=0.0205
🏃 View run mlp-hidden-64 at: http://localhost:5000/#/experiments/1/runs/080ec2daf36446f88340bd006a0dec6d
🧪 View experiment at: http://localhost:5000/#/experiments/1


/home/navadeep04/AIops/lib/python3.14/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


Logged run 044c50cfb10e4cf5ac552f02b762bbf3  |  acc=0.9651  f1=0.9647
🏃 View run mlp-hidden-256 at: http://localhost:5000/#/experiments/1/runs/044c50cfb10e4cf5ac552f02b762bbf3
🧪 View experiment at: http://localhost:5000/#/experiments/1
Logged run 6931d48a38ca46119285d94e8614b388  |  acc=0.6603  f1=0.6733
🏃 View run mlp-hidden-256 at: http://localhost:5000/#/experiments/1/runs/6931d48a38ca46119285d94e8614b388
🧪 View experiment at: http://localhost:5000/#/experiments/1
Logged run ef409a62742e43d499e2bb8657b8c84e  |  acc=0.1145  f1=0.0211
🏃 View run mlp-hidden-256 at: http://localhost:5000/#/experiments/1/runs/ef409a62742e43d499e2bb8657b8c84e
🧪 View experiment at: http://localhost:5000/#/experiments/1
Sweep run IDs: ['81460da545c24a63bcea2352c08fdd86', '7d8b6da57ba74ba48dea8523dca6553f', 'cfe4c4277e01430b9bd35284005139cb', 'eda80189780f42b3a5684683ab130b2e', '46e3047b7faf466d815766ee60ee916d', '080ec2daf36446f88340bd006a0dec6d', '044c50cfb10e4cf5ac552f02b762bbf3', '6931d48a38ca46119285d94

In [10]:
runs_df = mlflow.search_runs(
    experiment_names=["mnist-classifier"],
    order_by=["metrics.accuracy DESC"] )

display_cols = [c for c in runs_df.columns if c in (
    "run_id","tags.mlflow.runName","params.hidden_layer_sizes","params.learning_rate_init","params.max_iter","metrics.accuracy",
    "metrics.f1_macro")]

print(runs_df[display_cols].head(10).to_string(index=False))

best_run = runs_df.iloc[0]

print(f"\nBest run: {best_run['run_id']}  "
    f"(accuracy={best_run['metrics.accuracy']:.4f})")

                          run_id  metrics.f1_macro  metrics.accuracy params.max_iter params.hidden_layer_sizes tags.mlflow.runName
044c50cfb10e4cf5ac552f02b762bbf3          0.964746          0.965071              20                    (256,)      mlp-hidden-256
805ecf0bbf1d40a7aa4ef096e5570833          0.953907          0.954429              20                    (128,)        mlp-baseline
554684c1612b49f99e4f5075fc0c0848          0.953907          0.954429              20                    (128,)        mlp-baseline
eda80189780f42b3a5684683ab130b2e          0.945112          0.945571              20                     (64,)       mlp-hidden-64
81460da545c24a63bcea2352c08fdd86          0.930898          0.931857              20                     (32,)       mlp-hidden-32
46e3047b7faf466d815766ee60ee916d          0.851535          0.837286              20                     (64,)       mlp-hidden-64
6931d48a38ca46119285d94e8614b388          0.673257          0.660286              2